# NLP Assignent 1: Unfair Clause Detection

## Task 1

### Loading the (pre-downloaded) datasets

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('./data')

files = {
    'train': DATA_DIR / 'train.csv',
    'validation': DATA_DIR / 'validation.csv',
    'test': DATA_DIR / 'test.csv'
}

train_df = pd.read_csv(files['train'])
validation_df = pd.read_csv(files['validation'])
test_df = pd.read_csv(files['test'])

print('Training set:', train_df.shape)
print('Validation set:', validation_df.shape)
print('Test set:', test_df.shape)

display(test_df.head(5))

Training set: (15837, 8)
Validation set: (2548, 8)
Test set: (2032, 8)


,document_ID,document,text,A,CH,CR,LTD,TER
0,90,imgur,using our app to do anything other than access...,0,0,0,0,0
1,90,imgur,"if you do not consent , do not use our website .",0,0,0,0,0
2,90,imgur,your use of our website to do anything beyond ...,0,0,0,0,0
3,90,imgur,these terms of use are our contract with you .,0,0,0,0,0
4,90,imgur,"we also have community rules , developed thank...",0,0,0,0,0


### Removing unwanted columns

In [26]:
columns = [
    'document_ID',
    'text',
    'A',
    'CH',
    'CR',
    'LTD',
    'TER'
]

train_df = train_df[columns].copy()
validation_df = validation_df[columns].copy()
test_df = test_df[columns].copy()

display(test_df.head(5))

,document_ID,text,A,CH,CR,LTD,TER
0,90,using our app to do anything other than access...,0,0,0,0,0
1,90,"if you do not consent , do not use our website .",0,0,0,0,0
2,90,your use of our website to do anything beyond ...,0,0,0,0,0
3,90,these terms of use are our contract with you .,0,0,0,0,0
4,90,"we also have community rules , developed thank...",0,0,0,0,0


In [27]:
display(test_df.dtypes)

memory_test_before = test_df.memory_usage(deep=True).sum()
print(f'Label memory (test set): {memory_test_before / 1024:.2f} KB')


document_ID     int64
text           object
A               int64
CH              int64
CR              int64
LTD             int64
TER             int64
dtype: object

Label memory (test set): 591.49 KB


### Building the target

Even if it is not requested I decided to change the data-types of the labels from 'int64' to 'uint8' (Since the values are binary) in order to reduce memory usage.

*Note: If the cell is executed multiple times then the compared memory values change because the target column is added.*

In [28]:
label_columns = ['A', 'CH', 'CR', 'LTD', 'TER']
    
# Changing data types to 'uint8'
train_df[label_columns] = train_df[label_columns].astype('uint8')
validation_df[label_columns] = validation_df[label_columns].astype('uint8')
test_df[label_columns] = test_df[label_columns].astype('uint8')

memory_test_after = test_df.memory_usage(deep=True).sum()
print(f'Label memory (test set): from {memory_test_before / 1024:.2f} KB to {memory_test_after / 1024:.2f} KB')

# Creating the target
train_df['target'] = train_df[label_columns].values.tolist()
validation_df['target'] = validation_df[label_columns].values.tolist()
test_df['target'] = test_df[label_columns].values.tolist()

display(test_df.head(5))

Label memory (test set): from 591.49 KB to 522.04 KB


,document_ID,text,A,CH,CR,LTD,TER,target
0,90,using our app to do anything other than access...,0,0,0,0,0,"[0, 0, 0, 0, 0]"
1,90,"if you do not consent , do not use our website .",0,0,0,0,0,"[0, 0, 0, 0, 0]"
2,90,your use of our website to do anything beyond ...,0,0,0,0,0,"[0, 0, 0, 0, 0]"
3,90,these terms of use are our contract with you .,0,0,0,0,0,"[0, 0, 0, 0, 0]"
4,90,"we also have community rules , developed thank...",0,0,0,0,0,"[0, 0, 0, 0, 0]"


### Data inspection

#### Counting labels

First I check how many positive examples for each label are in the splits. Then I check the number of positive labels for each clause.

In [34]:
splits = {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}

# Positive examples for each label
print('Positive examples for each label')
positive_counts = pd.DataFrame({
    split_name: df[label_columns].sum()
    for split_name, df in splits.items()
})

positive_counts.index.name = "Label"

display(positive_counts)

# Positive labels per clause
print('Positive labels per clause')
for df in splits.values():
    df["num_labels"] = df[label_columns].sum(axis=1)

label_distribution = {}

for split_name, df in splits.items():

    label_distribution[split_name] = {
        "0 labels": (df["num_labels"] == 0).sum(),
        "1 label": (df["num_labels"] == 1).sum(),
        "2 labels": (df["num_labels"] == 2).sum(),
        "3 labels": (df["num_labels"] == 3).sum(),
        "4 labels": (df["num_labels"] == 4).sum(),
        "5 labels": (df["num_labels"] == 5).sum()
    }

distribution_df = pd.DataFrame(label_distribution)

display(distribution_df)

Positive examples for each label


,Train,Validation,Test
Label,,,
A,75,20,11
CH,268,43,33
CR,165,32,19
LTD,504,75,47
TER,318,59,43


Positive labels per clause


,Train,Validation,Test
0 labels,14614,2336,1893
1 label,1124,195,127
2 labels,91,17,10
3 labels,8,0,2
4 labels,0,0,0
5 labels,0,0,0


#### Checking co-occurrencies

In the end I wanted to check the co-occurrences of the labeles. The matrix displayed below shows an interesting fact: when a clause is labeled as 'A' no other label is ever positive (label 'A' is mutually exclusive with all other labels). Given the small number of examples in the dataset labeled with 'A' this could be happening by chance, but I will kep this fact in mind for later.

In [ ]:
cooccurrence_matrices = {}

for split_name, df in splits.items():

    labels = df[label_columns].to_numpy(dtype=np.int64)

    cooccurrence = labels.T @ labels

    cooccurrence_matrices[split_name] = pd.DataFrame(
        cooccurrence,
        index=label_columns,
        columns=label_columns
    )

# Display training co-occurrences
print('Co-occurrence (Train)')
display(cooccurrence_matrices["Train"])
print('Co-occurrence (Validation)')
display(cooccurrence_matrices["Validation"])
print('Co-occurrence (Test)')
display(cooccurrence_matrices["Test"])

Co-occurrence (Train)


,A,CH,CR,LTD,TER
A,75,0,0,0,0
CH,0,268,4,11,37
CR,0,4,165,5,44
LTD,0,11,5,504,14
TER,0,37,44,14,318


Co-occurrence (Validation)


,A,CH,CR,LTD,TER
A,20,0,0,0,0
CH,0,43,0,0,6
CR,0,0,32,2,9
LTD,0,0,2,75,0
TER,0,6,9,0,59


Co-occurrence (Test)


,A,CH,CR,LTD,TER
A,11,0,0,0,0
CH,0,33,2,2,4
CR,0,2,19,1,6
LTD,0,2,1,47,1
TER,0,4,6,1,43


## Task 2

### Task 2a - Data Cleaning

### Task 2b - Label Analysis